# University Analytics: Data Integration and Preprocessing
This notebook focuses strictly on merging datasets, deriving key features, handling missing values and outliers, and generating a final clean dataset in Excel form.

In [1]:
import pandas as pd
import numpy as np
import warnings
warnings.filterwarnings('ignore')


## 1. Data Loading and Merging

In [2]:
file_path = 'University_Management_Curation_Project.xlsx'
xl = pd.ExcelFile(file_path)

# Load sheets
students = xl.parse('students')
courses = xl.parse('courses')
enrollments = xl.parse('enrollments')
attendance = xl.parse('attendance')
grades = xl.parse('grades')
faculty = xl.parse('faculty')

print("Data loaded successfully.")


Data loaded successfully.


## 2. Standardize Inconsistent Grading Formats

In [3]:
# Assuming A, B, C, D, F, with F being fail. Maybe some numerical grades or P/F?
grade_mapping = {'A': 4.0, 'B': 3.0, 'C': 2.0, 'D': 1.0, 'F': 0.0, 'P': 4.0}

def standardize_grade(g):
    if pd.isna(g):
        return np.nan
    g_str = str(g).strip().upper()
    g_str = g_str.replace('+', '').replace('-', '')
    if g_str in grade_mapping:
        return grade_mapping[g_str]
    try:
        val = float(g_str)
        if val > 4.0:
            return min(val / 25.0, 4.0)
        return val
    except:
        return np.nan

grades['gpa_score'] = grades['grade'].apply(standardize_grade)

# Create 'pass' binary indicator
grades['pass'] = (grades['gpa_score'] >= 1.0).astype(int)
print("Grading formats standardized.")


Grading formats standardized.


## 3. Handle Missing Values and Outliers

In [4]:
# Impute missing values in grades using course medians
grades['gpa_score'] = grades.groupby('course_id')['gpa_score'].transform(lambda x: x.fillna(x.median()))
grades['gpa_score'] = grades['gpa_score'].fillna(grades['gpa_score'].median())

# Fill missing credits in courses with median if any exist
courses['credits'] = pd.to_numeric(courses['credits'], errors='coerce')
courses['credits'] = courses['credits'].fillna(courses['credits'].median())

print("Missing values handled.")


Missing values handled.


## 4. Feature Engineering

In [5]:
# Attendance Percentage
attendance['is_present'] = attendance['status'].apply(lambda x: 1 if str(x).lower().strip() in ['present', 'late', 'p'] else 0)
att_summary = attendance.groupby(['student_id', 'course_id']).agg(
    total_classes=('attendance_date', 'count'),
    attended_classes=('is_present', 'sum')
).reset_index()
att_summary['attendance_pct'] = (att_summary['attended_classes'] / att_summary['total_classes']) * 100

# Course Load per semester
course_load = enrollments.merge(courses[['course_id', 'credits']], on='course_id', how='left')
student_load = course_load.groupby(['student_id', 'semester']).agg(
    total_credits=('credits', 'sum')
).reset_index()

# Overall student GPA
student_gpa = grades.groupby('student_id')['gpa_score'].mean().reset_index().rename(columns={'gpa_score': 'overall_gpa'})

# Course Pass Rate
course_pass_rate = grades.groupby('course_id').agg(
    total_students=('student_id', 'nunique'),
    passed_students=('pass', 'sum')
).reset_index()
course_pass_rate['pass_rate'] = (course_pass_rate['passed_students'] / course_pass_rate['total_students']) * 100

print("Derived features: GPA, Attendance Percentage, Course load, Pass rate created.")


Derived features: GPA, Attendance Percentage, Course load, Pass rate created.


## 5. Merge Datasets and Handle Features Outliers

In [6]:
# Merge derived metrics onto the students base table
final_df = students.copy()
final_df = final_df.merge(student_gpa, on='student_id', how='left')

# Average attendance per student overall
avg_attendance = att_summary.groupby('student_id')['attendance_pct'].mean().reset_index().rename(columns={'attendance_pct': 'avg_attendance_pct'})
final_df = final_df.merge(avg_attendance, on='student_id', how='left')

# Average course load per student (across all semesters)
avg_load = student_load.groupby('student_id')['total_credits'].mean().reset_index().rename(columns={'total_credits': 'avg_course_load'})
final_df = final_df.merge(avg_load, on='student_id', how='left')

# Filling final missing values for students without grades/enrollments
final_df['overall_gpa'] = final_df['overall_gpa'].fillna(final_df['overall_gpa'].median())
final_df['avg_attendance_pct'] = final_df['avg_attendance_pct'].fillna(final_df['avg_attendance_pct'].median())
final_df['avg_course_load'] = final_df['avg_course_load'].fillna(final_df['avg_course_load'].median())

# Handle Outliers using IQR rule for attendance and course load
def cap_outliers(df, col):
    Q1 = df[col].quantile(0.25)
    Q3 = df[col].quantile(0.75)
    IQR = Q3 - Q1
    lower_bound = Q1 - 1.5 * IQR
    upper_bound = Q3 + 1.5 * IQR
    df[col] = np.clip(df[col], lower_bound, upper_bound)

cap_outliers(final_df, 'avg_attendance_pct')
cap_outliers(final_df, 'avg_course_load')

# Merging with faculty (Adding FacultyID to Course Pass Rates to demonstrate cross-table merging)
# courses table has faculty_id
course_with_faculty = courses[['course_id', 'faculty_id']].merge(course_pass_rate, on='course_id', how='left')
course_with_faculty['pass_rate'] = course_with_faculty['pass_rate'].fillna(0)

print("Final datasets prepared.")
final_df.head()


Final datasets prepared.


,student_id,name,dob,gender,email,phone,enrollment_year,department_id,overall_gpa,avg_attendance_pct,avg_course_load
0,S10000,Michelle Lamb,2004-09-13,Other,egreen@morgan.com,265.233.5210x4181,2018,D18,2.250000,33.333333,6.500000
1,S10001,Julie Hicks,1996-03-14,Male,wyoung@cobb-murphy.biz,001-192-564-6990x039,2020,D3,1.000000,66.666667,9.333333
2,S10002,Stanley Swanson,2004-05-13,Male,zduran@gmail.com,+1-956-544-0123x23423,2018,D3,2.000000,25.000000,4.500000
3,S10003,Melanie Gay,2005-09-18,Other,danacolon@jones.com,5428064334,2020,D8,2.666667,0.000000,5.000000
4,S10004,Jessica Shah,1999-08-09,Female,billytaylor@gmail.com,943-200-5504x485,2023,D4,3.000000,100.000000,5.500000


## 6. Export to Excel

In [7]:
# Save the processed data into an Excel file with multiple sheets
output_path = 'Processed_University_Data.xlsx'

with pd.ExcelWriter(output_path, engine='openpyxl') as writer:
    final_df.to_excel(writer, sheet_name='Student_Derived_Metrics', index=False)
    course_with_faculty.to_excel(writer, sheet_name='Course_Faculty_PassRates', index=False)
    
print(f"Data successfully exported to {output_path}")


Data successfully exported to Processed_University_Data.xlsx
